In [7]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# --- Initial setup (Mock for demonstration) ---
# Assuming 'spark' is an active SparkSession and 'cassandra_df' is the DataFrame
# you were trying to write, which contained extra columns (like endtime/lastupdatedtime).

# Replace with your actual SparkSession initialization
spark = (
    SparkSession.builder
    .appName("Cassandra Write Fix")
    .getOrCreate()
)

# Mock DataFrame that represents the data *before* writing to Cassandra
# This mock intentionally includes the extra columns 'endtime' and 'lastupdatedtime'
# that caused the original error.
mock_schema = StructType([
    StructField("pricearea", StringType(), True),
    StructField("productiongroup", StringType(), True),
    StructField("starttime", StringType(), True),
    StructField("quantitykwh", DoubleType(), True),
    # These two columns caused the error because they don't exist in Cassandra
    StructField("endtime", StringType(), True),
    StructField("lastupdatedtime", StringType(), True),
])

# Create a mock DataFrame with the problematic schema
cassandra_df_with_extra_cols = spark.createDataFrame([
    ("NO1", "P", "2021-01-01T00:00:00", 10.5, "2021-01-01T01:00:00", "2021-01-01T00:00:00"),
], schema=mock_schema)

print("--- Mock DataFrame Schema (Includes extra columns) ---")
cassandra_df_with_extra_cols.printSchema()

# -----------------------------------------------------------
# --- FIX: Explicitly select only the required columns ---
# -----------------------------------------------------------

# The required columns must match the Cassandra table schema exactly:
# pricearea, productiongroup, starttime, quantitykwh
REQUIRED_CASSANDRA_COLUMNS = [
    "pricearea",
    "productiongroup",
    "starttime",
    "quantitykwh",
]

# Create a new DataFrame containing only the columns that match the Cassandra table schema
df_to_write = cassandra_df_with_extra_cols.select(REQUIRED_CASSANDRA_COLUMNS)

print("\n--- Corrected DataFrame Schema (Matches Cassandra) ---")
df_to_write.printSchema()
print("\nAttempting to write with corrected column selection...")

# Replace 'df_to_write' with your actual DataFrame variable if applying this fix
# to your existing script. The actual write operation would now look like this:
try:
    (
        df_to_write.write
        .format("org.apache.spark.sql.cassandra")
        .mode("append") # or "overwrite"
        .option("keyspace", "elhub")
        .option("table", "production_2021")
        # Ensure the connection options are correctly set here as well
        # .option("confirm.truncate", "true") # Only if you use 'overwrite' mode
        .save()
    )
    print("✅ Successfully prepared DataFrame for writing to Cassandra.")
except Exception as e:
    # If the write were attempted, it would now likely proceed without the NoSuchElementException
    print(f"❌ Actual write error (e.g., connection issues): {e}")


# --- Optional: Further analysis of the original log warning ---
# The warning about 'PlainTextAuthProviderBase' suggests that your Cassandra instance
# might be configured to expect authentication (username/password), but the driver
# is not providing it, or the Cassandra connection is allowing the connection anyway.
# While the main error is schema-related, this warning indicates a potential
# security or configuration issue if you intended to use authentication.
print("\n--- Additional Log Analysis ---")
print("⚠️ WARN SparkContext: Another SparkContext is being constructed...")
print("    This is usually harmless in Jupyter/Databricks environments, but in a clean script, ensure you only create one SparkSession (or SparkContext).")
print("⚠️ WARN PlainTextAuthProviderBase: /127.0.0.1:9042 did not send an authentication challenge...")
print("    This suggests your Cassandra server expects authentication but the Spark connector settings might be missing the username/password, or Cassandra is configured to allow unauthenticated access despite expecting credentials.")

spark.stop()

25/11/14 22:21:12 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.command

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.net.BindException: Can't assign requested address: Service 'sparkDriver' failed after 16 retries (on a random free port)! Consider explicitly setting the appropriate binding address for the service 'sparkDriver' (for example spark.driver.bindAddress for SparkDriver) to the correct binding address.
	at java.base/sun.nio.ch.Net.bind0(Native Method)
	at java.base/sun.nio.ch.Net.bind(Net.java:555)
	at java.base/sun.nio.ch.ServerSocketChannelImpl.netBind(ServerSocketChannelImpl.java:337)
	at java.base/sun.nio.ch.ServerSocketChannelImpl.bind(ServerSocketChannelImpl.java:294)
	at io.netty.channel.socket.nio.NioServerSocketChannel.doBind(NioServerSocketChannel.java:141)
	at io.netty.channel.AbstractChannel$AbstractUnsafe.bind(AbstractChannel.java:562)
	at io.netty.channel.DefaultChannelPipeline$HeadContext.bind(DefaultChannelPipeline.java:1334)
	at io.netty.channel.AbstractChannelHandlerConte